# 02 – Spectrum preprocessing and dataset (Version 0.2)

Inspects the processed *E. coli* + ciprofloxacin dataset built by `scripts/build_dataset.py`. **No model is trained here.**

Build it first (from the project root):
```
python scripts/build_dataset.py
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import exploration as ex
from src.data_loader import read_raw_spectrum
from src.dataset import load_dataset, resolve_relpath
from src.preprocessing import PreprocessingConfig
from src.splits import load_split
from src.utils import driams_root, load_config, project_path

config = load_config(PROJECT_ROOT / "config.yaml")
pcfg = PreprocessingConfig.from_config(config)
data_dir = project_path(config["dataset"]["output_dir"]) / config["dataset"]["name"]
X, meta, summary = load_dataset(data_dir)   # X is memory-mapped, not copied into RAM
print(X.shape, X.dtype, summary['x_megabytes'], 'MB; rows fingerprint', summary['row_fingerprint'])

## 1. Preprocessing definition

In [ ]:
pd.Series(pcfg.to_dict())

## 2. Samples per site and exclusions

In [ ]:
display(pd.DataFrame(summary['per_site']).T[['target_species_rows', 'samples', 'resistant', 'susceptible', 'excluded']])
pd.DataFrame([{'reason': r, **c} for r, c in summary['exclusions_by_reason'].items()]).fillna(0)

## 3. One spectrum before and after preprocessing (no identifiers shown)

In [ ]:
ex.apply_style()
row = 0
r = meta.iloc[row]
raw = read_raw_spectrum(resolve_relpath(driams_root(config), r['spectrum_relpath']))
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(raw[:, 0], raw[:, 1], lw=0.8)
axes[0].set_title(f'Raw: {len(raw):,} points')
axes[1].plot(pcfg.bin_centers, X[row], lw=0.8)
axes[1].set_title(f'Processed: {X.shape[1]:,} bins, {np.count_nonzero(X[row]):,} non-zero, label {r.label}')
axes[1].set_xlabel('m/z (Da)')
fig.tight_layout()

## 4. Mean spectrum per class (descriptive only)

In [ ]:
labels = meta['label'].to_numpy()
mean_r = np.zeros(X.shape[1])
mean_s = np.zeros(X.shape[1])
for start in range(0, len(meta), 512):          # chunked, so the whole matrix is never loaded
    block = np.asarray(X[start:start + 512], dtype=np.float64)
    lab = labels[start:start + 512]
    mean_r += block[lab == 1].sum(axis=0)
    mean_s += block[lab == 0].sum(axis=0)
mean_r /= (labels == 1).sum()
mean_s /= (labels == 0).sum()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(pcfg.bin_centers, mean_s, lw=0.8, label='susceptible')
ax.plot(pcfg.bin_centers, mean_r, lw=0.8, label='resistant')
ax.set_xlim(2000, 12000)
ax.legend()
ax.set_title('Mean processed spectrum per class')

## 5. Splits (indices only; patient groups kept together)

In [ ]:
rows = []
for path in sorted((data_dir / 'splits').glob('*.json')):
    split = load_split(path, meta)   # checks the dataset fingerprint and sample / patient overlap
    s = split.summary(meta)
    for p in ('train', 'validation', 'test'):
        rows.append({'split': split.name, 'part': p,
                     **{k: s[p][k] for k in ('samples', 'resistant', 'susceptible', 'date_min', 'date_max')}})
pd.DataFrame(rows)